In [0]:
from pyspark.sql.functions import col, from_json, transform, filter, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

raw_df = spark.read.table("workspace.fotmob.raw_player_overview")

schema = StructType([
    StructField("name", StringType(), True),
    StructField("birthDate", StructType([
        StructField("utcTime", StringType(), True)
    ]), True),
    StructField("primaryTeam", StructType([
        StructField("teamId", IntegerType(), True),
        StructField("teamName", StringType(), True)
    ]), True),
    StructField("positionDescription", StructType([
        StructField("primaryPosition", StructType([
            StructField("key", StringType(), True),
            StructField("label", StringType(), True)
        ]), True),
        StructField("nonPrimaryPositions", ArrayType(StructType([
            StructField("key", StringType(), True),
            StructField("label", StringType(), True)
        ])), True)
    ]), True),
    StructField("playerInformation", ArrayType(StructType([
        StructField("title", StringType(), True),
        StructField("translationKey", StringType(), True),
        StructField("value", StructType([
            StructField("key", StringType(), True),
            StructField("fallback", StringType(), True),
            StructField("numberValue", IntegerType(), True)
        ]), True)
    ])), True)
])

df = raw_df.select(
    col("player_id"),
    from_json(col("raw_json"), schema).alias("parsed")
).select(
    col("player_id"),
    col("parsed.name").alias("player_name"),
    col("parsed.birthDate.utcTime").alias("birthdate"),
    col("parsed.primaryTeam.teamId").alias("club_id"),
    col("parsed.primaryTeam.teamName").alias("club"),
    col("parsed.positionDescription.primaryPosition.label").alias("primary_position"),
    transform(
        col("parsed.positionDescription.nonPrimaryPositions"),
        lambda x: x["label"]
    ).alias("secondary_positions"),
    expr("get(filter(parsed.playerInformation, x -> x.translationKey = 'preferred_foot'), 0).value.key").alias("preferred_foot"),
    expr("get(filter(parsed.playerInformation, x -> x.translationKey = 'country_sentencecase'), 0).value.fallback").alias("country")
)
display(df)

player_id,player_name,birthdate,club_id,club,primary_position,secondary_positions,preferred_foot,country
99767,Sophie Schmidt,1988-06-28T00:00:00.000Z,521233,Houston Dash,midfielder,null,right,Canada
99818,Marta,1986-02-19T00:00:00.000Z,728922,Orlando Pride,Striker,"List(Right Winger, Attacking Midfielder, Left Winger)",left,Brazil
121234,Emilie Haavi,1992-06-16T00:00:00.000Z,584402,Roma,Left Winger,List(Left Midfielder),right,Norway
180455,Kosovare Asllani,1989-07-29T00:00:00.000Z,1075419,London City Lionesses,Attacking Midfielder,"List(Central Midfielder, Striker)",right,Sweden
271109,Alexandra Popp,1991-04-06T00:00:00.000Z,394121,VfL Wolfsburg,Striker,"List(Central Midfielder, Attacking Midfielder, Right Winger)",left,Germany
271439,Samantha Kerr,1993-09-10T00:00:00.000Z,258661,Chelsea,Striker,List(),right,Australia
289679,Stine Ballisager,1994-01-03T00:00:00.000Z,671935,Bayern München,Center Back,List(Defensive Midfielder),right,Denmark
294952,Svenja Huth,1991-01-25T00:00:00.000Z,394121,VfL Wolfsburg,Right Winger,"List(Right Midfielder, Central Midfielder, Attacking Midfielder)",both,Germany
298603,Marta Torrejón,1990-02-27T00:00:00.000Z,401657,Barcelona,Center Back,List(Right Back),right,Spain
314998,Lisa Naalsund,1995-06-11T00:00:00.000Z,954396,Manchester United,Central Midfielder,"List(Defensive Midfielder, Attacking Midfielder)",right,Norway


In [0]:
from delta.tables import DeltaTable

# Save the processed data to a new table with upsert logic
table_name = "workspace.fotmob.player_overview_processed"

# Create table if it doesn't exist
if not spark.catalog.tableExists(table_name):
    df.write.saveAsTable(table_name)
    print(f"Table created: {table_name}")
else:
    # Merge: replace existing players (by player_id), insert new ones
    delta_table = DeltaTable.forName(spark, table_name)
    
    delta_table.alias("target").merge(
        df.alias("source"),
        "target.player_id = source.player_id"  
    ).whenMatchedUpdateAll(  
    ).whenNotMatchedInsertAll( 
    ).execute()
    
    print(f"Table merged successfully: {table_name}")

print(f"Total rows: {spark.read.table(table_name).count()}")

Table merged successfully: workspace.fotmob.player_overview_processed
Total rows: 1488
